In [ ]:
pip install kagglehub

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import kagglehub

## Pregunta Inicial
- **Hipótesis en Bruto**:
	- Quien tiene mayor porcentaje de victorias Strikers o Grapplers?
	- Quien tiene peleas mas largas Strikers o Grapplers?
	- Cual es el porcentaje de victorias por tipo de peleador? (Strikers Vs Grapplers)
	- Que marca la diferencia en los encuentros?
	- Con que atributos esta relacionada la apuesta del peleador?
- **Hipótesis Finales (Base)**:
	- Existe una relación ente las apuestas (Favorito Vs UnderDog) con la victoria?
	- De qué manera esta relacionado el arquetipo del peleador con el resultado de la pelea? (Tiempo, Manera de Finalización)
	- Existe un patrón que explique el fenómeno de "La Invasión Daguestani"?

### Descargando Dataset 

In [ ]:
path = kagglehub.dataset_download("mdabbert/ultimate-ufc-dataset")

print("Path to dataset files:", path)

## Funciones

In [ ]:
def generar_perfil_completo(nombre, df_ufc):
    mask = (df_ufc['R_fighter'] == nombre) | (df_ufc['B_fighter'] == nombre)
    participaciones = df_ufc[mask].copy()
    
    if participaciones.empty:
        return pd.Series([[], 0, 0, 0, 0, 0, 0, 0])

    indices_peleas = participaciones.index.tolist()
    total_fights = len(participaciones)
    
    wins, losses, draws = 0, 0, 0
    resultados_historicos = []

    for _, row in participaciones.iterrows():
        winner = row['Winner']
        if winner == 'Draw':
            res = 'D'; draws += 1
        elif (winner == 'Red' and row['R_fighter'] == nombre) or \
             (winner == 'Blue' and row['B_fighter'] == nombre):
            res = 'W'; wins += 1
        else:
            res = 'L'; losses += 1
        resultados_historicos.append(res)
    
    historial_rev = resultados_historicos[::-1]
    racha_w, racha_l, racha_d = 0, 0, 0
    
    if historial_rev:
        ultimo_res = historial_rev[0]
        for res in historial_rev:
            if res == ultimo_res:
                if res == 'W': racha_w += 1
                elif res == 'L': racha_l += 1
                elif res == 'D': racha_d += 1
            else:
                break
                
    return pd.Series([
        indices_peleas, total_fights, wins, losses, draws, 
        racha_w, racha_l, racha_d
    ])

In [ ]:
def clasificar_arquetipo_base(row):
    td = row['avg_TD_landed']
    sub = row['avg_SUB_ATT']
    sig = row['avg_SIG_STR_landed']

    # Perfeccionar clasificacion
    es_grappler = (td > 1.5) or (sub > 0.7)
    es_striker  = (sig > 15) and (td < 1.0)
    
    if es_grappler and not es_striker:
        return 'Grappler'
    elif es_striker and not es_grappler:
        return 'Striker'
    else:
        return 'Completo'

### Importando Bibliotecas

In [ ]:
import pandas as pd

### Cargando el DataFrame

In [ ]:
df_ufc = pd.read_csv(f"{path}/ufc-master.csv")

*Primera Vista*

In [ ]:
df_ufc.head()

In [ ]:
df_ufc.tail()

## 1. Sanity Check

In [ ]:
print(df_ufc.shape)

In [ ]:
df_ufc.info()

In [ ]:
df_ufc.columns

In [ ]:
df_ufc['R_fighter'].unique()

In [ ]:
df_ufc['B_fighter'].unique()

In [ ]:
df_ufc.describe()

## Conociendo los datos

In [ ]:
eda_cols_pelea = ['Winner', 'finish', 'finish_details', 'no_of_rounds', 'finish_round', 'finish_round_time', 'total_fight_time_secs', 'weight_class', 'gender', 'date', 'location', 'country']

In [ ]:
df_ufc[eda_cols_pelea].head()

In [ ]:
df_ufc[eda_cols_pelea].tail()

In [ ]:
df_ufc[eda_cols_pelea].dtypes

In [ ]:
df_ufc[eda_cols_pelea].isnull().sum()

In [ ]:
df_ufc[eda_cols_pelea].describe()

In [ ]:
df_ufc[eda_cols_pelea].value_counts()

In [ ]:
df_ufc[eda_cols_pelea].nunique()

In [ ]:
eda_cols_peleador = ['R_fighter', 'R_Height_cms', 'R_Reach_cms', 'R_Weight_lbs', 'R_age', 'R_Stance']

In [ ]:
df_ufc[eda_cols_peleador].dtypes

In [ ]:
df_ufc[eda_cols_peleador].isnull().sum()

In [ ]:
df_ufc[eda_cols_peleador].describe()

In [ ]:
df_ufc[eda_cols_peleador].value_counts()

In [ ]:
df_ufc[eda_cols_peleador].nunique()

In [ ]:
eda_cols_apuestas = ['R_fighter', 'R_odds', 'r_ko_odds', 'r_sub_odds', 'r_dec_odds']

In [ ]:
df_ufc[eda_cols_apuestas].dtypes

In [ ]:
df_ufc[eda_cols_apuestas].isnull().sum()

In [ ]:
df_ufc[eda_cols_apuestas].describe()

In [ ]:
df_ufc[eda_cols_apuestas].nunique()

In [ ]:
eda_cols_acciones = ['R_fighter', 'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct', 'R_avg_SUB_ATT', 'R_avg_TD_landed', 'R_avg_TD_pct']

In [ ]:
df_ufc[eda_cols_acciones].dtypes

In [ ]:
df_ufc[eda_cols_acciones].isnull().sum()

In [ ]:
df_ufc[eda_cols_acciones].describe()

In [ ]:
df_ufc[eda_cols_acciones].value_counts()

In [ ]:
df_ufc[eda_cols_acciones].nunique()

## Cual es el periodo de tiempo en el que se encuentran nuestros datos?

In [ ]:
df_ufc['date'].describe()

In [ ]:
df_ufc['date'] = pd.to_datetime(df_ufc['date'])

In [ ]:
print(df_ufc['date'].dtype)

In [ ]:
df_ufc = df_ufc.sort_values(by='date', ascending=True).reset_index(drop=True)

In [ ]:
df_ufc['date'].head()

In [ ]:
df_ufc['date'].tail()

Van desde **(2010-03-21 hasta 2026-03-28)**

## Atributos de Acciones

In [ ]:
df_ufc[['B_avg_SIG_STR_landed', 'R_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct', 'R_avg_SIG_STR_pct', 'B_avg_SUB_ATT', 'R_avg_SUB_ATT', 'B_avg_TD_landed', 'R_avg_TD_landed', 'B_avg_TD_pct', 'R_avg_TD_pct']].describe()

### Separar en DF's para recuperar todas las estadisticas del peleador (Acciones y Caracteristicas)

In [ ]:
cols_interes = [
    'fighter', 'avg_SIG_STR_landed', 'avg_SIG_STR_pct', 
    'avg_SUB_ATT', 'avg_TD_landed', 'avg_TD_pct', 
    'Height_cms', 'Reach_cms', 'Weight_lbs', 'age', 'Stance'
]

### DF's (Red y Blue)

In [ ]:
df_red = df_ufc[['R_' + col for col in cols_interes]].copy()
df_red.columns = cols_interes

In [ ]:
df_blue = df_ufc[['B_' + col for col in cols_interes]].copy()
df_blue.columns = cols_interes

### DF Red

In [ ]:
df_red.head()

In [ ]:
df_red.tail()

In [ ]:
df_red.info()

### DF Blue

In [ ]:
df_blue.head()

In [ ]:
df_blue.tail()

In [ ]:
df_blue.info()

### Unimos los DF's en uno solo

In [ ]:
df_red_blue = pd.concat([df_red, df_blue], axis=0, ignore_index=True)

In [ ]:
df_red_blue.head()

In [ ]:
df_red_blue.tail()

In [ ]:
df_red_blue.info()

In [ ]:
df_red_blue['fighter'].unique()

In [ ]:
df_red_blue.dropna()

In [ ]:
df_red_blue.loc[df_red_blue['fighter'] == 'Ilia Topuria']

### Separamos el DF unificado (Red y Blue) en DF's especificos (Acciones y Caractersiticas)

In [ ]:
df_peleador_acciones = df_red_blue.groupby('fighter').agg({
    'avg_SIG_STR_landed': 'mean',
    'avg_SIG_STR_pct': 'mean',
    'avg_SUB_ATT': 'mean',
    'avg_TD_landed': 'mean',
    'avg_TD_pct': 'mean'
}).reset_index()

In [ ]:
df_peleador_caracteristicas = df_red_blue.groupby('fighter').agg({
    'Height_cms': 'last',
    'Reach_cms': 'last',
    'Weight_lbs': 'last',
    'age': 'max',
    'Stance': 'last'
}).reset_index()

In [ ]:
df_peleador_acciones['arquetipo_base'] = df_peleador_acciones.apply(
    clasificar_arquetipo_base, axis=1
)

In [ ]:
peleas_red = df_ufc.groupby('R_fighter').apply(lambda x: x.index.tolist()).to_dict()
peleas_blue = df_ufc.groupby('B_fighter').apply(lambda x: x.index.tolist()).to_dict()

In [ ]:
df_peleador_record = df_red_blue[['fighter']].drop_duplicates().reset_index(drop=True)

In [ ]:
columnas_nuevas = [
    'indices_peleas', 'total_fights', 'wins', 'losses', 'draws', 
    'current_win_streak', 'current_loss_streak', 'current_draw_streak'
]

In [ ]:
df_peleador_record[columnas_nuevas] = df_peleador_record['fighter'].apply(
    lambda x: generar_perfil_completo(x, df_ufc)
)

### DF Acciones 

In [ ]:
df_peleador_acciones = df_peleador_acciones.dropna(subset=['fighter'])

In [ ]:
df_peleador_acciones.shape

In [ ]:
df_peleador_acciones.head()

In [ ]:
df_peleador_acciones.tail()

In [ ]:
df_peleador_acciones.describe()

### Peleadores especificos

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Ilia Topuria']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Diego Lopes']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Georges St-Pierre']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Jon Jones']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Conor McGregor']

### DF Caracteristicas

In [ ]:
df_peleador_caracteristicas = df_peleador_caracteristicas.dropna(subset=['fighter'])

In [ ]:
df_peleador_caracteristicas.shape

In [ ]:
df_peleador_caracteristicas.head()

In [ ]:
df_peleador_caracteristicas.tail()

In [ ]:
df_peleador_caracteristicas.describe()

### Peleador especificos

In [ ]:
df_peleador_caracteristicas.loc[df_peleador_caracteristicas['fighter'] == 'Georges St-Pierre']

## DF Record

In [ ]:
df_peleador_record.head()

In [ ]:
df_peleador_record.describe()

In [ ]:
df_peleador_record.tail()

### Peleadores especificos

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Ilia Topuria']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Tony Ferguson']

### Accedemos a la información de las peleas del peleador

In [ ]:
cols_finish = [
    'R_fighter', 'B_fighter', 'Winner', 'finish', 'finish_details', 'finish_round', 'finish_round_time'
]

In [ ]:
cols_odds = [
    'R_fighter', 'B_fighter', 'Winner', 'finish', 'finish_details',
    'R_odds', 'B_odds', 'r_ko_odds','b_ko_odds', 'r_sub_odds', 'b_sub_odds'
]

In [ ]:
cols_acciones = [
    'R_fighter', 'B_fighter', 'Winner', 'finish', 'finish_details',
    'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct', 'R_avg_SUB_ATT', 'R_avg_TD_landed', 'R_avg_TD_pct',
    'B_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct', 'B_avg_SUB_ATT', 'B_avg_TD_landed', 'B_avg_TD_pct'
]

In [ ]:
nombre = 'Diego Lopes'
indices = df_peleador_record.loc[
    df_peleador_record['fighter'] == nombre, 'indices_peleas'
].values[0]

df_ufc.loc[indices, cols_finish]

In [ ]:
nombre = 'Diego Lopes'
indices = df_peleador_record.loc[
    df_peleador_record['fighter'] == nombre, 'indices_peleas'
].values[0]

df_ufc.loc[indices, cols_odds]

In [ ]:
nombre = 'Diego Lopes'
indices = df_peleador_record.loc[
    df_peleador_record['fighter'] == nombre, 'indices_peleas'
].values[0]

df_ufc.loc[indices, cols_acciones]

## Primeros graficos

In [ ]:
df_arq_time = df_ufc[['total_fight_time_secs', 'Winner', 'R_fighter', 'B_fighter']].copy()

# Realizar esto en el principal
df_arq_time = df_arq_time.merge(
    df_peleador_acciones[['fighter', 'arquetipo_base']],
    left_on='R_fighter', right_on='fighter', how='left'
).rename(columns={'arquetipo_base': 'arquetipo_red'}).drop(columns='fighter')

df_arq_time = df_arq_time.merge(
    df_peleador_acciones[['fighter', 'arquetipo_base']],
    left_on='B_fighter', right_on='fighter', how='left'
).rename(columns={'arquetipo_base': 'arquetipo_blue'}).drop(columns='fighter')

df_arq_time['arquetipo_winner'] = df_arq_time.apply(
    lambda row: row['arquetipo_red'] if row['Winner'] == 'Red' 
    else row['arquetipo_blue'] if row['Winner'] == 'Blue' 
    else None, axis=1
)

df_arq_time['arquetipo_losser'] = df_arq_time.apply(
    lambda row: row['arquetipo_red'] if row['Winner'] == 'Blue' 
    else row['arquetipo_blue'] if row['Winner'] == 'Red' 
    else None, axis=1
)

In [ ]:
df_arq_time.head()

In [ ]:
df_arq_time['arquetipo_winner'].describe()

In [ ]:
df_arq_time.describe()

In [ ]:
plt.style.use('_mpl-gallery')

arquetipos = ['Striker', 'Grappler', 'All-Around']
data = [
    df_arq_time[df_arq_time['arquetipo_winner'] == arq]['total_fight_time_secs'].dropna()
    for arq in arquetipos
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(data, labels=arquetipos, patch_artist=True)
ax.set_xlabel('Arquetipo')
ax.set_ylabel('Duración de pelea (segundos)')
ax.set_title('Duración de peleas por arquetipo')
plt.show()

In [ ]:
plt.style.use('_mpl-gallery')

arquetipos = ['Striker', 'Grappler', 'All-Around']
data = [
    df_arq_time[df_arq_time['arquetipo_losser'] == arq]['total_fight_time_secs'].dropna()
    for arq in arquetipos
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(data, labels=arquetipos, patch_artist=True)
ax.set_xlabel('Arquetipo')
ax.set_ylabel('Duración de pelea (segundos)')
ax.set_title('Duración de peleas por arquetipo')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colores = {'Striker': 'steelblue', 'Grappler': 'tomato', 'All-Around': 'seagreen'}

for arq, color in colores.items():
    subset = df_arq_time[df_arq_time['arquetipo_winner'] == arq]['total_fight_time_secs'].dropna()
    ax.hist(subset, bins=30, alpha=0.5, label=arq, color=color, density=True)

# Líneas de round
for round_num in range(1, 6):
    ax.axvline(x=round_num * 300, color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax.text(round_num * 300 + 5, ax.get_ylim()[1] * 0.95, f'R{round_num}', 
            color='red', fontsize=8)

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x//60)}:{int(x%60):02d}'))
ax.set_xlabel('Duración (min:seg)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de duración de peleas ganadas por arquetipo')
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colores = {'Striker': 'steelblue', 'Grappler': 'tomato', 'All-Around': 'seagreen'}

for arq, color in colores.items():
    subset = df_arq_time[df_arq_time['arquetipo_losser'] == arq]['total_fight_time_secs'].dropna()
    ax.hist(subset, bins=30, alpha=0.5, label=arq, color=color, density=True)

# Líneas de round
for round_num in range(1, 6):
    ax.axvline(x=round_num * 300, color='red', linestyle='--', alpha=0.6, linewidth=1)
    ax.text(round_num * 300 + 5, ax.get_ylim()[1] * 0.95, f'R{round_num}', 
            color='red', fontsize=8)

ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x//60)}:{int(x%60):02d}'))
ax.set_xlabel('Duración (min:seg)')
ax.set_ylabel('Densidad')
ax.set_title('Distribución de duración de peleas perdidas por arquetipo')
ax.legend()
plt.show()

In [ ]:
df_record_arq = df_peleador_record.merge(
    df_peleador_acciones[['fighter', 'arquetipo_base']],
    on='fighter', how='left'
).dropna(subset=['arquetipo_base'])

# 2. Calcular win_rate por arquetipo
df_record_arq['win_rate'] = df_record_arq['wins'] / df_record_arq['total_fights']
df_record_arq['loss_rate'] = df_record_arq['losses'] / df_record_arq['total_fights']
df_record_arq['draw_rate'] = df_record_arq['draws'] / df_record_arq['total_fights']

# 3. Agrupar por arquetipo
df_arq_stats = df_record_arq.groupby('arquetipo_base').agg(
    win_rate=('win_rate', 'mean'),
    loss_rate=('loss_rate', 'mean'),
    draw_rate=('draw_rate', 'mean'),
    n_fighters=('fighter', 'count')
).reset_index()

print(df_arq_stats)  # Validar antes de graficar

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

arquetipos = df_arq_stats['arquetipo_base']
x = np.arange(len(arquetipos))
width = 0.5

colores = {
    'win_rate':  'seagreen',
    'loss_rate': 'tomato',
    'draw_rate': 'steelblue'
}

bars_w = ax.bar(x, df_arq_stats['win_rate'],  width, label='Victorias', color=colores['win_rate'])
bars_l = ax.bar(x, df_arq_stats['loss_rate'], width, label='Derrotas',  color=colores['loss_rate'],
                bottom=df_arq_stats['win_rate'])
bars_d = ax.bar(x, df_arq_stats['draw_rate'], width, label='Empates',   color=colores['draw_rate'],
                bottom=df_arq_stats['win_rate'] + df_arq_stats['loss_rate'])

# Etiquetas de % dentro de cada barra
for i, row in df_arq_stats.iterrows():
    # Win
    ax.text(i, row['win_rate'] / 2,
            f"{row['win_rate']:.1%}", ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
    # Loss
    ax.text(i, row['win_rate'] + row['loss_rate'] / 2,
            f"{row['loss_rate']:.1%}", ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')

# N de peleadores bajo cada barra
for i, row in df_arq_stats.iterrows():
    ax.text(i, -0.08, f"n={int(row['n_fighters'])}",
            ha='center', va='top', fontsize=9, color='gray')

ax.set_xticks(x)
ax.set_xticklabels(arquetipos, fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylim(0, 1.05)
ax.set_ylabel('Porcentaje')
ax.set_title('Win Rate promedio por Arquetipo', fontsize=13, fontweight='bold')
ax.legend(loc='upper right')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
def contar_finalizaciones(nombre, indices, df_ufc):
    if not indices:
        return pd.Series({'total_ko': 0, 'total_sub': 0, 'total_dec': 0})
    
    peleas = df_ufc.loc[indices, ['finish', 'Winner', 'R_fighter', 'B_fighter']]
    
    ko = sub = dec = 0
    
    for _, row in peleas.iterrows():
        gano = (row['Winner'] == 'Red'  and row['R_fighter'] == nombre) or \
               (row['Winner'] == 'Blue' and row['B_fighter'] == nombre)
        
        if not gano:
            continue

        f = str(row['finish']).strip()
        
        # Ajusta estos valores según el print de arriba
        if f in ['KO/TKO', 'TKO', 'TKO - Doctor\'s Stop']:
            ko  += 1
        elif f in ['SUB', 'Submission']:
            sub += 1
        elif f in ['DEC', 'S-DEC', 'M-DEC', 'U-DEC']:
            dec += 1
        #else:
            # Ver qué casos no estamos capturando
            #print(f"Sin clasificar → {nombre}: '{f}'")
    
    return pd.Series({'total_ko': ko, 'total_sub': sub, 'total_dec': dec})

In [ ]:
df_fin = df_peleador_record.apply(
    lambda row: contar_finalizaciones(row['fighter'], row['indices_peleas'], df_ufc),
    axis=1
)

df_peleador_record = pd.concat([df_peleador_record, df_fin], axis=1)

In [ ]:
test = ['Conor McGregor', 'Khabib Nurmagomedov', 'Jon Jones']
df_peleador_record[
    df_peleador_record['fighter'].isin(test)
][['fighter', 'wins', 'total_ko', 'total_sub', 'total_dec']]

In [ ]:
# ── DF BASE ────────────────────────────────────────────────────────
df_base = df_peleador_record.merge(
    df_peleador_acciones[['fighter', 'arquetipo_base']],
    on='fighter', how='left'
).dropna(subset=['arquetipo_base'])

df_base['ko_rate']  = df_base['total_ko']  / df_base['wins']
df_base['sub_rate'] = df_base['total_sub'] / df_base['wins']
df_base['dec_rate'] = df_base['total_dec'] / df_base['wins']

# Agrupar por arquetipo
finish_arq = df_base.groupby('arquetipo_base').agg(
    ko_rate  = ('ko_rate',  'mean'),
    sub_rate = ('sub_rate', 'mean'),
    dec_rate = ('dec_rate', 'mean'),
    n        = ('fighter',  'count')
).reset_index()

print(finish_arq)

In [ ]:
arquetipos = finish_arq['arquetipo_base'].tolist()
x = np.arange(len(arquetipos))
w = 0.5

fig, ax = plt.subplots(figsize=(8, 5))

b1 = ax.bar(x, finish_arq['ko_rate'],  w, label='KO/TKO',  color='#378ADD')
b2 = ax.bar(x, finish_arq['sub_rate'], w, label='Sumisión', color='#1D9E75',
            bottom=finish_arq['ko_rate'])
b3 = ax.bar(x, finish_arq['dec_rate'], w, label='Decisión', color='#888780',
            bottom=finish_arq['ko_rate'] + finish_arq['sub_rate'])

# Etiquetas dentro de cada segmento
for i, row in finish_arq.iterrows():
    # KO
    if row['ko_rate'] > 0.03:
        ax.text(i, row['ko_rate'] / 2,
                f"{row['ko_rate']:.1%}", ha='center', va='center',
                fontsize=10, color='white', fontweight='500')
    # SUB
    if row['sub_rate'] > 0.03:
        ax.text(i, row['ko_rate'] + row['sub_rate'] / 2,
                f"{row['sub_rate']:.1%}", ha='center', va='center',
                fontsize=10, color='white', fontweight='500')
    # DEC
    if row['dec_rate'] > 0.03:
        ax.text(i, row['ko_rate'] + row['sub_rate'] + row['dec_rate'] / 2,
                f"{row['dec_rate']:.1%}", ha='center', va='center',
                fontsize=10, color='white', fontweight='500')
    # N peleadores
    ax.text(i, -0.10, f"n={int(row['n'])}",
            ha='center', va='top', fontsize=9, color='gray')

ax.set_xticks(x)
ax.set_xticklabels(arquetipos, fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylim(0, 1.1)
ax.set_ylabel('% de victorias')
ax.set_title('¿Cómo terminan sus peleas los diferentes arquetipos?', fontweight='500')
ax.legend(loc='upper right')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Verificar nombre exacto de la columna
print(df_ufc['finish'].unique())
print(df_ufc['weight_class'].unique())

# ── DF DIVISIÓN ────────────────────────────────────────────────────
orden_divisiones = [
    "Women's Strawweight", "Women's Flyweight", "Women's Bantamweight",
    "Strawweight", "Flyweight", "Bantamweight", "Featherweight",
    "Lightweight", "Welterweight", "Middleweight",
    "Light Heavyweight", "Heavyweight"
]

# Filtrar solo divisiones presentes en el dataset
orden_divisiones = [d for d in orden_divisiones if d in df_ufc['weight_class'].unique()]

labels_cortos = {
    "Women's Strawweight": "W.Straw", "Women's Flyweight": "W.Fly",
    "Women's Bantamweight": "W.Bantam", "Strawweight": "Straw",
    "Flyweight": "Fly",               "Bantamweight": "Bantam",
    "Featherweight": "Feather",       "Lightweight": "Light",
    "Welterweight": "Welter",         "Middleweight": "Middle",
    "Light Heavyweight": "LHW",       "Heavyweight": "Heavy"
}

df_div = df_ufc[['weight_class', 'total_fight_time_secs', 'finish']].dropna()

div_stats = df_div.groupby('weight_class').agg(
    mediana_secs = ('total_fight_time_secs', 'median'),
    media_secs   = ('total_fight_time_secs', 'mean'),
    ko_pct       = ('finish', lambda x: (x.isin(['KO/TKO', 'TKO', "TKO - Doctor's Stop"])).mean()),
    sub_pct      = ('finish', lambda x: (x.isin(['SUB', 'Submission'])).mean()),
    dec_pct      = ('finish', lambda x: (x.isin(['DEC', 'S-DEC', 'M-DEC', 'U-DEC'])).mean()),
    n_peleas     = ('finish', 'count')
).reindex(orden_divisiones).dropna()

print(div_stats)

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))
x = np.arange(len(div_stats))
xtick_labels = [labels_cortos.get(d, d) for d in div_stats.index]
w = 0.6

ax1.bar(x, div_stats['mediana_secs'], w, color='#378ADD', alpha=0.5, label='Mediana')
ax1.bar(x, div_stats['media_secs'],   w * 0.4, color='#378ADD', alpha=0.9, label='Media')

for i, (_, row) in enumerate(div_stats.iterrows()):
    mins = int(row['mediana_secs'] // 60)
    secs = int(row['mediana_secs'] % 60)
    ax1.text(i, row['mediana_secs'] + 8, f"{mins}:{secs:02d}",
             ha='center', va='bottom', fontsize=12, color='#185FA5')

for r in range(1, 6):
    ax1.axhline(r * 300, color='red', linestyle='--', alpha=0.25, linewidth=0.9)
    ax1.text(len(x) - 0.45, r * 300 + 6, f'R{r}',
             color='red', fontsize=7, alpha=0.5)

for i, (_, row) in enumerate(div_stats.iterrows()):
    ax1.text(i, 0, f"n={int(row['n_peleas'])}",
             ha='center', va='bottom', fontsize=7, color='black')

ax1.set_xticks(x)
ax1.set_xticklabels(xtick_labels, fontsize=9, rotation=20, ha='right')
ax1.set_ylabel('Duración (min:seg)')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(
    lambda y, _: f"{int(y//60)}:{int(y%60):02d}"))
ax1.set_title('Duración de peleas por división de peso', fontweight='500')
ax1.legend(fontsize=9)
ax1.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax2 = plt.subplots(figsize=(12, 5))

ax2.bar(x, div_stats['ko_pct'],  w, label='KO/TKO',  color='#378ADD')
ax2.bar(x, div_stats['sub_pct'], w, label='Sumisión', color='#1D9E75',
        bottom=div_stats['ko_pct'])
ax2.bar(x, div_stats['dec_pct'], w, label='Decisión', color='#888780',
        bottom=div_stats['ko_pct'] + div_stats['sub_pct'])

for i, (_, row) in enumerate(div_stats.iterrows()):
    if row['ko_pct'] > 0.05:
        ax2.text(i, row['ko_pct'] / 2,
                 f"{row['ko_pct']:.0%}", ha='center', va='center',
                 fontsize=8, color='white', fontweight='500')
    if row['sub_pct'] > 0.05:
        ax2.text(i, row['ko_pct'] + row['sub_pct'] / 2,
                 f"{row['sub_pct']:.0%}", ha='center', va='center',
                 fontsize=8, color='white', fontweight='500')
    if row['dec_pct'] > 0.05:
        ax2.text(i, row['ko_pct'] + row['sub_pct'] + row['dec_pct'] / 2,
                 f"{row['dec_pct']:.0%}", ha='center', va='center',
                 fontsize=8, color='white', fontweight='500')

ax2.set_xticks(x)
ax2.set_xticklabels(xtick_labels, fontsize=9, rotation=20, ha='right')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.set_ylabel('% de peleas')
ax2.set_title('Manera de finalización por división de peso', fontweight='500')
ax2.legend(loc='upper left', fontsize=9)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Merge record + arquetipos + acciones
df_analisis = df_peleador_record.merge(
    df_peleador_acciones[['fighter', 'arquetipo_base', 
                           'avg_SIG_STR_landed', 'avg_SIG_STR_pct',
                           'avg_SUB_ATT', 'avg_TD_landed', 'avg_TD_pct']],
    on='fighter', how='left'
).dropna(subset=['arquetipo_base'])

# 2. Promedio de acciones por arquetipo
df_acciones_arq = df_analisis.groupby('arquetipo_base').agg(
    sig_str  = ('avg_SIG_STR_landed', 'mean'),
    str_pct  = ('avg_SIG_STR_pct',    'mean'),
    sub_att  = ('avg_SUB_ATT',        'mean'),
    td_land  = ('avg_TD_landed',      'mean'),
    td_pct   = ('avg_TD_pct',         'mean'),
).reset_index()

print(df_acciones_arq)

In [ ]:

# 1. Agrupar métricas por arquetipo
df_acciones_arq = df_analisis.groupby('arquetipo_base').agg(
    sig_str = ('avg_SIG_STR_landed', 'mean'),
    str_pct = ('avg_SIG_STR_pct',    'mean'),
    sub_att = ('avg_SUB_ATT',        'mean'),
    td_land = ('avg_TD_landed',      'mean'),
    td_pct  = ('avg_TD_pct',         'mean'),
).reset_index()

# 2. Separar métricas por tipo (volumen vs precisión)
metricas_volumen   = ['sig_str', 'sub_att', 'td_land']
metricas_precision = ['str_pct', 'td_pct']

labels_volumen   = ['Sig. Strikes\nlanzados', 'Intentos\nsumisión', 'Takedowns\nlanzados']
labels_precision = ['Precisión\nstrike %', 'Precisión\ntakedown %']

arquetipos = df_acciones_arq['arquetipo_base'].tolist()
colores    = {'Striker': '#378ADD', 'Grappler': '#D85A30', 'Completo': '#1D9E75'}

x_vol = np.arange(len(metricas_volumen))
x_pre = np.arange(len(metricas_precision))
width = 0.25

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# --- Panel izquierdo: Volumen ---
for i, arq in enumerate(arquetipos):
    row = df_acciones_arq[df_acciones_arq['arquetipo_base'] == arq].iloc[0]
    vals = [row[m] for m in metricas_volumen]
    offset = (i - 1) * width
    bars = ax1.bar(x_vol + offset, vals, width, label=arq,
                   color=colores[arq], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.05,
                 f'{v:.1f}', ha='center', va='bottom', fontsize=8)

ax1.set_xticks(x_vol)
ax1.set_xticklabels(labels_volumen, fontsize=10)
ax1.set_ylabel('Promedio por pelea')
ax1.set_title('Acciones — volumen', fontsize=11)
ax1.spines[['top', 'right']].set_visible(False)
ax1.legend(fontsize=9)

# --- Panel derecho: Precisión ---
for i, arq in enumerate(arquetipos):
    row = df_acciones_arq[df_acciones_arq['arquetipo_base'] == arq].iloc[0]
    vals = [row[m] for m in metricas_precision]
    offset = (i - 1) * width
    bars = ax2.bar(x_pre + offset, vals, width, label=arq,
                   color=colores[arq], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.005,
                 f'{v:.1%}', ha='center', va='bottom', fontsize=8)

ax2.set_xticks(x_pre)
ax2.set_xticklabels(labels_precision, fontsize=10)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax2.set_title('Acciones — precisión', fontsize=11)
ax2.spines[['top', 'right']].set_visible(False)
ax2.legend(fontsize=9)

fig.suptitle('¿Qué acciones ejecuta más cada arquetipo?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(df_ufc['title_bout'].value_counts())

In [ ]:
cols_perf = [
    'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct',
    'R_avg_TD_landed',       'R_avg_TD_pct',
    'R_avg_SUB_ATT',
    'B_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct',
    'B_avg_TD_landed',       'B_avg_TD_pct',
    'B_avg_SUB_ATT',
    'title_bout'
]

df_titulo = df_ufc[cols_perf].copy()

df_red_t = df_titulo[['R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct',
                       'R_avg_TD_landed', 'R_avg_TD_pct', 
                       'R_avg_SUB_ATT', 'title_bout']].rename(columns=lambda c: c.replace('R_', ''))

df_blue_t = df_titulo[['B_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct',
                        'B_avg_TD_landed', 'B_avg_TD_pct',
                        'B_avg_SUB_ATT', 'title_bout']].rename(columns=lambda c: c.replace('B_', ''))

df_long_titulo = pd.concat([df_red_t, df_blue_t], ignore_index=True)
df_long_titulo['tipo'] = df_long_titulo['title_bout'].map({True: 'Título', False: 'Normal'})

df_comp = df_long_titulo.groupby('tipo').agg(
    sig_str = ('avg_SIG_STR_landed', 'mean'),
    str_pct = ('avg_SIG_STR_pct',    'mean'),
    td_land = ('avg_TD_landed',      'mean'),
    td_pct  = ('avg_TD_pct',         'mean'),
    sub_att = ('avg_SUB_ATT',        'mean'),
).reset_index()

print(df_comp)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metricas = {
    'Sig. Strikes\nLanzados' : 'sig_str',
    'Takedowns\nLanzados'    : 'td_land',
    'Intentos\nSumisión'     : 'sub_att',
}
colores_tipo = {'Título': 'gold', 'Normal': 'steelblue'}

for ax, (label, col) in zip(axes, metricas.items()):
    for i, (_, row) in enumerate(df_comp.iterrows()):
        bar = ax.bar(i, row[col], color=colores_tipo[row['tipo']], 
                     width=0.5, label=row['tipo'])
        ax.text(i, row[col] + 0.05, f"{row[col]:.2f}", 
                ha='center', fontsize=10, fontweight='bold')
    
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Título', 'Normal'])
    ax.set_title(label, fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_ylim(0, df_comp[col].max() * 1.25)

fig.suptitle('Desempeño en Peleas por Título vs Normales', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Analizando UFC 328

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Grant Dawson']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Grant Dawson']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Mateusz Rebecki']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Mateusz Rebecki']

Considero ganará **Grant Dawson**

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Joel Alvarez']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Joel Alvarez']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Yaroslav Amosov']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Yaroslav Amosov']

Considero ganará **Joel Alvarez**

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Sean Brady']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Sean Brady']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Joaquin Buckley']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Joaquin Buckley']

Viendo las estadisticas creo que ganará **Sean Brady**

In [ ]:
busq = df_peleador_acciones[df_peleador_acciones['fighter'].str.contains('Alek', na=False)]
print(busq)

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Alekander Volkov']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Alekander Volkov']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Waldo Cortes-Acosta']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Waldo Cortes-Acosta']

No conocemos lo suficiente de **Alekander Volkov** como para hacer un analisis.

In [ ]:
df_peleador_caracteristicas.loc[df_peleador_caracteristicas['fighter'] == 'Joshua Van']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Joshua Van']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Joshua Van']

In [ ]:
nombre = 'Joshua Van'
indices = df_peleador_record.loc[
    df_peleador_record['fighter'] == nombre, 'indices_peleas'
].values[0]

df_ufc.loc[indices, cols_finish]

In [ ]:
df_peleador_caracteristicas.loc[df_peleador_caracteristicas['fighter'] == 'Tatsuro Taira']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Tatsuro Taira']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Tatsuro Taira']

In [ ]:
nombre = 'Tatsuro Taira'
indices = df_peleador_record.loc[
    df_peleador_record['fighter'] == nombre, 'indices_peleas'
].values[0]

df_ufc.loc[indices, cols_finish]

Viendo las estadisticas creo que ganará **Tatsuro Taira**

In [ ]:
df_peleador_caracteristicas.loc[df_peleador_caracteristicas['fighter'] == 'Khamzat Chimaev']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Khamzat Chimaev']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Khamzat Chimaev']

In [ ]:
nombre = 'Khamzat Chimaev'
indices = df_peleador_record.loc[
    df_peleador_record['fighter'] == nombre, 'indices_peleas'
].values[0]

df_ufc.loc[indices, cols_finish]

In [ ]:
df_peleador_caracteristicas.loc[df_peleador_caracteristicas['fighter'] == 'Sean Strickland']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Sean Strickland']

In [ ]:
df_peleador_record.loc[df_peleador_record['fighter'] == 'Sean Strickland']

In [ ]:
nombre = 'Sean Strickland'
indices = df_peleador_record.loc[
    df_peleador_record['fighter'] == nombre, 'indices_peleas'
].values[0]

df_ufc.loc[indices, cols_finish]

Viendo las estadisticas creo que ganará **Khamzat Chimaev** por sumisión en el tercer round.

| Mi favorito a ganar | Resultado | Información |
| :--- | :--- | :--- |
| Grant Dawson | Gana por sumisión - R3 04:42 | |
| Joel Alvarez | Pierde por sumisión - R2 01:13 | No teniamos suficiente información del rival para un analisis preciso |
| Sean Brady | Gana por decisión - R3 05:00 | Dominó al rival toda la pelea con su estilo de Grappler |
| Tatsuro Taira | Pierde por nocaut - R5 01:32 | Estuvo cerca de someter pero tuvo muchos problemas con los golpes del rival |
| Khamzat Chimaev | Pierde por decisión - R5 5:00 | No llevó la pelea como se esperaba, su rival aguantó sus intentos de derribo y al pelear de pie ambos Khamzat perdió por decisión |


In [ ]:
busq = df_peleador_acciones[df_peleador_acciones['fighter'].str.contains('Charles', na=False)]
print(busq)

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Charles Oliveira']

## Analisis Visual

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Donald Cerrone']

In [ ]:
df_peleador_caracteristicas.loc[df_peleador_caracteristicas['fighter'] == 'Donald Cerrone']

In [ ]:
df_peleador_acciones.loc[df_peleador_acciones['fighter'] == 'Melsik Baghdasaryan']

In [ ]:
df_peleador_caracteristicas.loc[df_peleador_acciones['fighter'] == 'Melsik Baghdasaryan']

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# 1. Preparar datos por arquetipo
df_acciones_arq = df_analisis.groupby('arquetipo_base').agg(
    sig_str = ('avg_SIG_STR_landed', 'mean'),
    str_pct = ('avg_SIG_STR_pct',    'mean'),
    sub_att = ('avg_SUB_ATT',        'mean'),
    td_land = ('avg_TD_landed',      'mean'),
    td_pct  = ('avg_TD_pct',         'mean'),
).reset_index()

# 2. Normalizar 0-1 para que las métricas sean comparables
cols_norm = ['sig_str', 'str_pct', 'sub_att', 'td_land', 'td_pct']
df_norm = df_acciones_arq.copy()
for col in cols_norm:
    min_v = df_norm[col].min()
    max_v = df_norm[col].max()
    df_norm[col] = (df_norm[col] - min_v) / (max_v - min_v)

# 3. Config
categorias = [
    'Sig. Strikes<br>lanzados',
    'Precisión<br>strike',
    'Intentos<br>sumisión',
    'Takedowns<br>lanzados',
    'Precisión<br>takedown'
]

colores = {
    'Striker'  : ('#378ADD', 'rgba(55,138,221,0.15)'),
    'Grappler' : ('#D85A30', 'rgba(216,90,48,0.15)'),
    'Completo' : ('#1D9E75', 'rgba(29,158,117,0.15)'),
}

# 4. Construir figura
fig = go.Figure()

for _, row in df_norm.iterrows():
    arq = row['arquetipo_base']
    vals_norm = [row[c] for c in cols_norm]
    
    # Valores reales para el tooltip
    vals_real = df_acciones_arq[
        df_acciones_arq['arquetipo_base'] == arq
    ][cols_norm].values[0]
    
    labels_tooltip = [
        f'{cat.replace("<br>", " ")}: {real:.2f}'
        for cat, real in zip(categorias, vals_real)
    ]
    
    color_line, color_fill = colores[arq]
    
    fig.add_trace(go.Scatterpolar(
        r    = vals_norm + [vals_norm[0]],   # cerrar polígono
        theta= categorias + [categorias[0]],
        name = arq,
        text = labels_tooltip + [labels_tooltip[0]],
        hovertemplate = '%{text}<extra></extra>',
        line = dict(color=color_line, width=2.5),
        fill = 'toself',
        fillcolor = color_fill,
        marker = dict(size=6, color=color_line),
    ))

# 5. Layout
fig.update_layout(
    polar=dict(
        bgcolor='rgba(0,0,0,0)',
        radialaxis=dict(
            visible=True,
            range=[0, 1],
            tickvals=[0.25, 0.5, 0.75, 1.0],
            ticktext=['25%', '50%', '75%', '100%'],
            tickfont=dict(size=9, color='gray'),
            gridcolor='rgba(128,128,128,0.2)',
            linecolor='rgba(128,128,128,0.2)',
        ),
        angularaxis=dict(
            tickfont=dict(size=11),
            gridcolor='rgba(128,128,128,0.2)',
            linecolor='rgba(128,128,128,0.3)',
        ),
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.15,
        xanchor='center',
        x=0.5,
        font=dict(size=12),
    ),
    title=dict(
        text='Perfil de acciones por arquetipo',
        x=0.5,
        font=dict(size=15),
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    height=520,
    margin=dict(t=80, b=80, l=80, r=80),
)

fig.show()

In [ ]:
# En Streamlit
peleadores = st.multiselect(
    'Selecciona peleadores',
    options=df_peleador_acciones['fighter'].tolist(),
    default=['Khabib Nurmagomedov', 'Conor McGregor']
)

In [ ]:
# Normalización global — cada métrica se normaliza contra el máximo del dataset completo
# Calcular una sola vez fuera del gráfico
maximos_globales = {col: df_peleador_acciones[col].max() for col in cols_norm}

df_norm = df_sel.copy()
for col in cols_norm:
    max_global = maximos_globales[col]
    df_norm[col] = df_norm[col] / max_global if max_global > 0 else 0

In [ ]:
import plotly.graph_objects as go

peleadores = ['Charles Oliveira'
              , 'Georges St-Pierre', 'Conor McGregor'
             ]

df_sel = df_peleador_acciones[
    df_peleador_acciones['fighter'].isin(peleadores)
].copy()

cols_norm = ['avg_SIG_STR_landed', 'avg_SIG_STR_pct', 
             'avg_SUB_ATT', 'avg_TD_landed', 'avg_TD_pct']

categorias = [
    'Sig. Strikes<br>lanzados',
    'Precisión<br>strike',
    'Intentos<br>sumisión',
    'Takedowns<br>lanzados',
    'Precisión<br>takedown'
]

# ✅ Calcular máximos globales UNA SOLA VEZ, fuera del loop
maximos_globales = {col: df_peleador_acciones[col].max() for col in cols_norm}

df_norm = df_sel.copy()
for col in cols_norm:
    max_global = maximos_globales[col]
    # ✅ Siempre normalizar contra el máximo global — sin importar cuántos peleadores
    df_norm[col] = df_norm[col] / max_global if max_global > 0 else 0

colores = [
    ('#378ADD', 'rgba(55,138,221,0.10)'),
    ('#D85A30', 'rgba(216,90,48,0.10)'),
    ('#1D9E75', 'rgba(29,158,117,0.10)'),
    ('#9B59B6', 'rgba(155,89,182,0.10)'),
    ('#F39C12', 'rgba(243,156,18,0.10)'),
]

fig = go.Figure()

for i, (_, row_norm) in enumerate(df_norm.iterrows()):
    nombre   = row_norm['fighter']
    row_real = df_sel[df_sel['fighter'] == nombre].iloc[0]
    
    vals_norm = [row_norm[c] for c in cols_norm]
    tooltip   = [
        f'{cat.replace("<br>", " ")}: {row_real[col]:.2f}'
        for cat, col in zip(categorias, cols_norm)
    ]
    
    color_line, color_fill = colores[i % len(colores)]
    
    fig.add_trace(go.Scatterpolar(
        r             = vals_norm + [vals_norm[0]],
        theta         = categorias + [categorias[0]],
        name          = nombre,
        text          = tooltip + [tooltip[0]],
        hovertemplate = '%{text}<extra></extra>',
        line          = dict(color=color_line, width=2),
        fill          = 'toself',
        fillcolor     = color_fill,
        marker        = dict(size=5, color=color_line),
    ))

fig.update_layout(
    polar=dict(
        bgcolor='rgba(0,0,0,0)',
        radialaxis=dict(
            visible  = True,
            range    = [0, 1],
            tickvals = [0.25, 0.5, 0.75, 1.0],
            ticktext = ['25%', '50%', '75%', '100%'],
            tickfont = dict(size=9, color='gray'),
            gridcolor= 'rgba(128,128,128,0.2)',
            linecolor= 'rgba(128,128,128,0.2)',
        ),
        angularaxis=dict(
            tickfont = dict(size=11),
            gridcolor= 'rgba(128,128,128,0.2)',
            linecolor= 'rgba(128,128,128,0.3)',
        ),
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.20,
        xanchor='center',
        x=0.5,
        font=dict(size=11),
    ),
    title=dict(
        text='Perfil de acciones por peleador',
        x=0.5,
        font=dict(size=15),
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor ='rgba(0,0,0,0)',
    height=560,
    margin=dict(t=80, b=100, l=80, r=80),
)

fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Seleccionar peleadores
peleadores = ['Jon Jones', 'Georges St-Pierre', 'Khabib Nurmagomedov', 'Conor McGregor']

# 2. Extraer finalizaciones por peleador desde df_ufc
def get_finalizaciones(nombre, df_ufc):
    mask = (df_ufc['R_fighter'] == nombre) | (df_ufc['B_fighter'] == nombre)
    peleas = df_ufc[mask][['finish', 'Winner', 'R_fighter', 'B_fighter']].copy()
    
    # Solo peleas donde ganó
    peleas_ganadas = peleas[
        ((peleas['Winner'] == 'Red')  & (peleas['R_fighter'] == nombre)) |
        ((peleas['Winner'] == 'Blue') & (peleas['B_fighter'] == nombre))
    ]
    
    return peleas_ganadas['finish'].value_counts()

# 3. Colores por tipo de finalización
color_map = {
    'KO/TKO' : '#D85A30',
    'SUB'     : '#378ADD',
    'DEC'     : '#1D9E75',
    'S-DEC'   : '#9B59B6',
    'M-DEC'   : '#F39C12',
    'U-DEC'   : '#1D9E75',
    'CNC'     : '#888780',
    'DQ'      : '#888780',
}

# 4. Subplots — una torta por peleador
n     = len(peleadores)
cols  = 2
rows  = (n + 1) // cols

specs = [[{'type': 'pie'} for _ in range(cols)] for _ in range(rows)]

fig = make_subplots(
    rows=rows, cols=cols,
    specs=specs,
    subplot_titles=peleadores,
)

for i, nombre in enumerate(peleadores):
    row = i // cols + 1
    col = i %  cols + 1
    
    finalizaciones = get_finalizaciones(nombre, df_ufc)
    
    if finalizaciones.empty:
        continue
    
    labels = finalizaciones.index.tolist()
    values = finalizaciones.values.tolist()
    colors = [color_map.get(l, '#888780') for l in labels]
    
    fig.add_trace(
        go.Pie(
            labels           = labels,
            values           = values,
            name             = nombre,
            marker           = dict(colors=colors, line=dict(color='rgba(0,0,0,0.2)', width=1)),
            textinfo         = 'label+percent',
            hovertemplate    = '<b>%{label}</b><br>Victorias: %{value}<br>%{percent}<extra>' + nombre + '</extra>',
            hole             = 0.35,       # donut — más legible
            textfont         = dict(size=11),
            insidetextorientation='radial',
        ),
        row=row, col=col
    )

# 5. Layout
fig.update_layout(
    title=dict(
        text='Finalizaciones por peleador<br><sup>Solo victorias — % sobre total de wins</sup>',
        x=0.5,
        font=dict(size=15),
    ),
    showlegend   = False,
    paper_bgcolor= 'rgba(0,0,0,0)',
    plot_bgcolor = 'rgba(0,0,0,0)',
    height       = rows * 320,
    margin       = dict(t=100, b=40, l=40, r=40),
)

fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Seleccionar peleadores
peleadores = ['Jon Jones', 'Georges St-Pierre', 'Khabib Nurmagomedov', 'Conor McGregor']

# 2. Finalizaciones por las que PERDIÓ
def get_derrotas(nombre, df_ufc):
    mask = (df_ufc['R_fighter'] == nombre) | (df_ufc['B_fighter'] == nombre)
    peleas = df_ufc[mask][['finish', 'Winner', 'R_fighter', 'B_fighter']].copy()
    
    # ✅ Solo peleas donde perdió (invertido respecto al gráfico anterior)
    peleas_perdidas = peleas[
        ((peleas['Winner'] == 'Blue') & (peleas['R_fighter'] == nombre)) |
        ((peleas['Winner'] == 'Red')  & (peleas['B_fighter'] == nombre))
    ]
    
    return peleas_perdidas['finish'].value_counts()

# 3. Colores
color_map = {
    'KO/TKO' : '#D85A30',
    'SUB'     : '#378ADD',
    'DEC'     : '#1D9E75',
    'S-DEC'   : '#9B59B6',
    'M-DEC'   : '#F39C12',
    'U-DEC'   : '#1D9E75',
    'CNC'     : '#888780',
    'DQ'      : '#888780',
}

# 4. Subplots
n    = len(peleadores)
cols = 2
rows = (n + 1) // cols

specs = [[{'type': 'pie'} for _ in range(cols)] for _ in range(rows)]

fig = make_subplots(
    rows=rows, cols=cols,
    specs=specs,
    subplot_titles=peleadores,
)

for i, nombre in enumerate(peleadores):
    row = i // cols + 1
    col = i %  cols + 1
    
    derrotas = get_derrotas(nombre, df_ufc)
    
    # ✅ Manejar peleadores invictos
    if derrotas.empty:
        fig.add_trace(
            go.Pie(
                labels        = ['Sin derrotas'],
                values        = [1],
                name          = nombre,
                marker        = dict(colors=['#1D9E75']),
                textinfo      = 'label',
                hovertemplate = f'<b>{nombre}</b> no tiene derrotas registradas<extra></extra>',
                hole          = 0.35,
            ),
            row=row, col=col
        )
        continue
    
    labels = derrotas.index.tolist()
    values = derrotas.values.tolist()
    colors = [color_map.get(l, '#888780') for l in labels]
    
    fig.add_trace(
        go.Pie(
            labels                  = labels,
            values                  = values,
            name                    = nombre,
            marker                  = dict(colors=colors, line=dict(color='rgba(0,0,0,0.2)', width=1)),
            textinfo                = 'label+percent',
            hovertemplate           = '<b>%{label}</b><br>Derrotas: %{value}<br>%{percent}<extra>' + nombre + '</extra>',
            hole                    = 0.35,
            textfont                = dict(size=11),
            insidetextorientation   = 'radial',
        ),
        row=row, col=col
    )

# 5. Layout
fig.update_layout(
    title=dict(
        text='Finalizaciones por las que perdieron<br><sup>Solo derrotas — % sobre total de losses</sup>',
        x=0.5,
        font=dict(size=15),
    ),
    showlegend    = False,
    paper_bgcolor = 'rgba(0,0,0,0)',
    plot_bgcolor  = 'rgba(0,0,0,0)',
    height        = rows * 320,
    margin        = dict(t=100, b=40, l=40, r=40),
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# 1. Configuración
peleadores = ['Khabib Nurmagomedov', 'Conor McGregor']
metrica    = 'td_landed'  # opciones: 'sig_str', 'td_landed', 'sig_str_pct', 'sub_att'

metrica_config = {
    'sig_str' : {'col': 'sig_str',   'label': 'Sig. Strikes por minuto'},
    'td_landed'  : {'col': 'td_landed',    'label': 'Takedowns lanzados'},
    'sig_str_pct': {'col': 'sig_str_pct',  'label': 'Precisión de strikes %'},
    'sub_att'    : {'col': 'sub_att',      'label': 'Intentos de sumisión'},
}

# color por peleador, forma por resultado
colores_peleador = {
    peleadores[0]: '#378ADD',
    peleadores[1]: '#D85A30',
    #peleadores[2]: '#1D9E75',
    #peleadores[3]: '#9B59B6',
}

symbol_map = {'Victoria': 'circle', 'Derrota': 'x', 'Empate': 'diamond'}
size_map   = {'Victoria': 10,       'Derrota': 13,  'Empate': 10}

# 2. Extraer historial
def get_historial_scatter(nombre, df_ufc):
    mask   = (df_ufc['R_fighter'] == nombre) | (df_ufc['B_fighter'] == nombre)
    peleas = df_ufc[mask].copy()

    registros = []
    for n_pelea, (_, row) in enumerate(peleas.iterrows(), start=1):
        winner = row['Winner']
        if winner == 'Draw':
            resultado = 'Empate'
        elif (winner == 'Red'  and row['R_fighter'] == nombre) or \
             (winner == 'Blue' and row['B_fighter'] == nombre):
            resultado = 'Victoria'
        else:
            resultado = 'Derrota'

        lado     = 'R' if row['R_fighter'] == nombre else 'B'
        dur_secs = row.get('total_fight_time_secs',        None)
        sig_str  = row.get(f'{lado}_avg_SIG_STR_landed',   None)
        sig_pct  = row.get(f'{lado}_avg_SIG_STR_pct',      None)
        td_land  = row.get(f'{lado}_avg_TD_landed',         None)
        sub_att  = row.get(f'{lado}_avg_SUB_ATT',           None)


        registros.append({
            'peleador'   : nombre,
            'n_pelea'    : n_pelea,
            'resultado'  : resultado,
            'dur_min'    : dur_secs / 60 if dur_secs else None,
            'sig_str' : sig_str,
            'sig_str_pct': sig_pct,
            'td_landed'  : td_land,
            'sub_att'    : sub_att,
            'finish'     : row.get('finish', 'N/A'),
            'oponente'   : row['B_fighter'] if lado == 'R' else row['R_fighter'],
            'fecha'      : row['date'].strftime('%Y-%m-%d') if pd.notna(row['date']) else 'N/A',
        })

    return pd.DataFrame(registros)

# 3. Construir df combinado
col_x    = metrica_config[metrica]['col']
label_x  = metrica_config[metrica]['label']

df_todos = pd.concat([
    get_historial_scatter(p, df_ufc) for p in peleadores
], ignore_index=True).dropna(subset=[col_x, 'dur_min'])

# 4. Figura — un trace por peleador × resultado
fig = go.Figure()

# ✅ Trazar en orden: primero victorias, luego empates, luego derrotas
# para que las derrotas (X) queden encima visualmente
orden_resultado = ['Victoria', 'Empate', 'Derrota']

for resultado in orden_resultado:
    for nombre in peleadores:
        grupo = df_todos[
            (df_todos['peleador']  == nombre) &
            (df_todos['resultado'] == resultado)
        ]

        if grupo.empty:
            continue

        color = colores_peleador[nombre]

        fig.add_trace(go.Scatter(
            x    = grupo['dur_min'],
            y    = grupo[col_x],
            mode = 'markers',
            # ✅ nombre en leyenda solo en primera aparición del peleador
            name            = nombre if resultado == 'Victoria' else None,
            legendgroup     = nombre,
            showlegend      = resultado == 'Victoria',
            marker = dict(
                color   = color,
                symbol  = symbol_map[resultado],
                size    = size_map[resultado],
                opacity = 0.80,
                line    = dict(
                    width = 1.5,
                    color = 'rgba(0,0,0,0.25)'
                ),
            ),
            customdata    = grupo[['oponente', 'fecha', 'finish', 'resultado', 'n_pelea']].values,
            hovertemplate = (
                f'<b>{nombre}</b> vs %{{customdata[0]}}<br>'
                'Fecha       : %{customdata[1]}<br>'
                'Resultado   : <b>%{customdata[3]}</b><br>'
                'Finalización: %{customdata[2]}<br>'
                f'{label_x}: ' + '%{y:.2f}<br>'
                'Duración    : %{x:.1f} min<br>'
                'Pelea #%{customdata[4]}'
                '<extra></extra>'
            ),
        ))

# 5. Líneas de rounds en eje X
for r in range(1, 6):
    fig.add_vline(
        x                   = r * 5,
        line_dash           = 'dot',
        line_color          = 'rgba(128,128,128,0.25)',
        line_width          = 1,
        annotation_text     = f'R{r}',
        annotation_position = 'top',
        annotation_font     = dict(size=9, color='gray'),
    )

# 6. Promedio global de la métrica
avg = df_todos[col_x].mean()
fig.add_hline(
    y                   = avg,
    line_dash           = 'dash',
    line_color          = 'rgba(128,128,128,0.4)',
    line_width          = 1,
    annotation_text     = f'Promedio global: {avg:.2f}',
    annotation_position = 'right',
    annotation_font     = dict(size=9, color='gray'),
)

# 7. Leyenda de formas — anotación manual
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(symbol='circle', size=9, color='gray'),
    name='● Victoria', showlegend=True, legendgroup='shapes'
))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(symbol='x', size=11, color='gray'),
    name='✕ Derrota', showlegend=True, legendgroup='shapes'
))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode='markers',
    marker=dict(symbol='diamond', size=9, color='gray'),
    name='◆ Empate', showlegend=True, legendgroup='shapes'
))

# 8. Layout
fig.update_layout(
    title=dict(
        text=(
            'Historial comparado de peleadores'
            f'<br><sup>X = Duración · Y = {label_x} · Color = Peleador · Forma = Resultado</sup>'
        ),
        x    = 0.5,
        font = dict(size=14),
    ),
    xaxis=dict(
        title     = 'Duración (minutos)',
        gridcolor = 'rgba(128,128,128,0.15)',
        zeroline  = False,
        range     = [0, 27],
    ),
    yaxis=dict(
        title     = label_x,
        gridcolor = 'rgba(128,128,128,0.15)',
        zeroline  = False,
    ),
    legend=dict(
        orientation = 'v',
        yanchor     = 'top',
        y           = 1,
        xanchor     = 'left',
        x           = 1.02,
        font        = dict(size=11),
        tracegroupgap = 12,
    ),
    paper_bgcolor = 'rgba(0,0,0,0)',
    plot_bgcolor  = 'rgba(0,0,0,0)',
    height        = 560,
    margin        = dict(t=100, b=60, l=60, r=160),
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# 1. Configuración
peleadores = ['Diego Lopes']
inversion  = 100

# ✅ Selector de tipo de odds
tipo_odds = 'sub_odds'  # opciones: 'R_odds', 'r_dec_odds', 'r_sub_odds', 'r_ko_odds'

odds_config = {
    'odds'    : {'col_r': 'R_odds',     'col_b': 'B_odds',     'label': 'Odds generales'},
    'dec_odds': {'col_r': 'r_dec_odds', 'col_b': 'b_dec_odds', 'label': 'Odds decisión'},
    'sub_odds': {'col_r': 'r_sub_odds', 'col_b': 'b_sub_odds', 'label': 'Odds sumisión'},
    'ko_odds' : {'col_r': 'r_ko_odds',  'col_b': 'b_ko_odds',  'label': 'Odds KO/TKO'},
}

col_r  = odds_config[tipo_odds]['col_r']
col_b  = odds_config[tipo_odds]['col_b']
label  = odds_config[tipo_odds]['label']

colores_peleador = {
    peleadores[0]: '#378ADD',
    #peleadores[1]: '#D85A30',
    #peleadores[2]: '#1D9E75',
    #peleadores[3]: '#9B59B6',
}

symbol_map = {'Victoria': 'circle', 'Derrota': 'x',  'Empate': 'diamond'}
size_map   = {'Victoria': 10,       'Derrota': 13,    'Empate': 10}

orden_tipo = ['Gran favorito', 'Favorito', "Pick'em", 'Underdog', 'Gran underdog']
color_tipo = {
    'Gran favorito' : '#1D9E75',
    'Favorito'      : '#378ADD',
    "Pick'em"       : '#888780',
    'Underdog'      : '#F39C12',
    'Gran underdog' : '#D85A30',
}

# 2. Funciones
def calcular_retorno(odds, inversion, gano):
    if not gano:
        return -inversion
    if pd.isna(odds):
        return None
    if odds > 0:
        ganancia = inversion * (odds / 100)
    else:
        ganancia = inversion * (100 / abs(odds))
    return round(ganancia, 2)

def get_tipo_apuesta(odds):
    if pd.isna(odds):
        return None
    elif odds < -200:
        return 'Gran favorito'
    elif odds < 0:
        return 'Favorito'
    elif odds < 150:
        return "Pick'em"
    elif odds < 300:
        return 'Underdog'
    else:
        return 'Gran underdog'

# 3. Extraer historial — ahora recibe col_r y col_b dinámicamente
def get_historial_apuestas(nombre, df_ufc, inversion, col_r, col_b):
    mask   = (df_ufc['R_fighter'] == nombre) | (df_ufc['B_fighter'] == nombre)
    peleas = df_ufc[mask].copy()

    registros = []
    for n_pelea, (_, row) in enumerate(peleas.iterrows(), start=1):
        lado = 'R' if row['R_fighter'] == nombre else 'B'

        winner = row['Winner']
        if winner == 'Draw':
            resultado = 'Empate'
            gano      = False
        elif (winner == 'Red'  and lado == 'R') or \
             (winner == 'Blue' and lado == 'B'):
            resultado = 'Victoria'
            gano      = True
        else:
            resultado = 'Derrota'
            gano      = False

        # ✅ Seleccionar columna correcta según lado y tipo de odds elegido
        odds    = row.get(col_r if lado == 'R' else col_b, None)
        retorno = calcular_retorno(odds, inversion, gano)
        tipo    = get_tipo_apuesta(odds)
        oponente= row['B_fighter'] if lado == 'R' else row['R_fighter']
        fecha   = row['date'].strftime('%Y-%m-%d') if pd.notna(row['date']) else 'N/A'

        registros.append({
            'peleador' : nombre,
            'n_pelea'  : n_pelea,
            'resultado': resultado,
            'odds'     : odds,
            'tipo'     : tipo,
            'retorno'  : retorno,
            'oponente' : oponente,
            'fecha'    : fecha,
            'finish'   : row.get('finish', 'N/A'),
        })

    return pd.DataFrame(registros)

# 4. Construir df combinado
df_todos = pd.concat([
    get_historial_apuestas(p, df_ufc, inversion, col_r, col_b)
    for p in peleadores
], ignore_index=True).dropna(subset=['retorno', 'tipo'])

df_todos['tipo'] = pd.Categorical(df_todos['tipo'], categories=orden_tipo, ordered=True)

# 5. Figura
fig = go.Figure()

for resultado in ['Victoria', 'Empate', 'Derrota']:
    for nombre in peleadores:
        grupo = df_todos[
            (df_todos['peleador']  == nombre) &
            (df_todos['resultado'] == resultado)
        ]
        if grupo.empty:
            continue

        fig.add_trace(go.Scatter(
            x    = grupo['retorno'],
            y    = grupo['tipo'],
            mode = 'markers',
            name            = nombre if resultado == 'Victoria' else None,
            legendgroup     = nombre,
            showlegend      = resultado == 'Victoria',
            marker = dict(
                color   = colores_peleador[nombre],
                symbol  = symbol_map[resultado],
                size    = size_map[resultado],
                opacity = 0.80,
                line    = dict(width=1.5, color='rgba(0,0,0,0.2)'),
            ),
            customdata    = grupo[['oponente', 'fecha', 'finish', 'resultado', 'odds', 'n_pelea']].values,
            hovertemplate = (
                f'<b>{nombre}</b> vs %{{customdata[0]}}<br>'
                'Fecha         : %{customdata[1]}<br>'
                'Resultado     : <b>%{customdata[3]}</b><br>'
                'Finalización  : %{customdata[2]}<br>'
                f'Odds ({label}): %{{customdata[4]}}<br>'
                'Retorno neto  : $%{x:.2f}<br>'
                'Pelea #%{customdata[5]}'
                '<extra></extra>'
            ),
        ))

# 6. Referencias
fig.add_vline(
    x=0, line_dash='dash', line_color='rgba(128,128,128,0.5)', line_width=1.5,
    annotation_text='Break-even', annotation_position='top',
    annotation_font=dict(size=9, color='gray'),
)
fig.add_vline(
    x=-inversion, line_dash='dot', line_color='rgba(216,90,48,0.4)', line_width=1,
    annotation_text=f'-${inversion}', annotation_position='bottom',
    annotation_font=dict(size=9, color='#D85A30'),
)

# 7. Leyenda de formas
for sym, lbl in [('circle','Victoria'), ('x','Derrota'), ('diamond','Empate')]:
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode='markers',
        marker=dict(symbol=sym, size=9, color='gray'),
        name=lbl, showlegend=True, legendgroup='shapes'
    ))

# 8. Layout
fig.update_layout(
    title=dict(
        text=(
            f'ROI por tipo de apuesta — {label}<br>'
            f'<sup>Inversión ${inversion} · X = Retorno neto · Y = Tipo de apuesta · Color = Peleador · Forma = Resultado</sup>'
        ),
        x=0.5, font=dict(size=14),
    ),
    xaxis=dict(
        title     = f'Retorno neto (USD) sobre ${inversion} invertidos',
        gridcolor = 'rgba(128,128,128,0.15)',
        zeroline  = False,
        tickprefix= '$',
    ),
    yaxis=dict(
        title         = 'Tipo de apuesta',
        gridcolor     = 'rgba(128,128,128,0.15)',
        categoryorder = 'array',
        categoryarray = orden_tipo,
    ),
    legend=dict(
        orientation='v', yanchor='top', y=1,
        xanchor='left', x=1.02,
        font=dict(size=11), tracegroupgap=12,
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor ='rgba(0,0,0,0)',
    height=500,
    margin=dict(t=110, b=60, l=120, r=160),
)

fig.show()

In [ ]:
print([c for c in df_ufc.columns if 'weight' in c.lower() or 'class' in c.lower() or 'divis' in c.lower()])
print(df_ufc.iloc[0][['R_fighter', 'B_fighter', 'date']])

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# 1. Configuración
col_division = 'weight_class'
col_genero   = 'gender'
ventana      = 20

# ✅ Selector
metrica = 'dur_min'  # opciones abajo

metrica_config = {
    'dur_min'    : {'col': 'dur_min',     'label': 'Duración (minutos)'},
    'sig_str_r'  : {'col': 'sig_str_r',   'label': 'Sig. Strikes lanzados (Red)'},
    'sig_pct_r'  : {'col': 'sig_pct_r',   'label': 'Precisión strikes (Red)'},
    'td_land_r'  : {'col': 'td_land_r',   'label': 'Takedowns lanzados (Red)'},
    'sub_att_r'  : {'col': 'sub_att_r',   'label': 'Intentos sumisión (Red)'},
}

col_sel   = metrica_config[metrica]['col']
label_sel = metrica_config[metrica]['label']

# 2. Preparar datos — extraer todas las métricas de una vez
df_plot = df_ufc[[
    col_division, col_genero, 'date',
    'total_fight_time_secs',
    'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct',
    'R_avg_TD_landed', 'R_avg_SUB_ATT',
]].copy()

df_plot = df_plot.dropna(subset=[col_division, col_genero])
df_plot['dur_min']   = df_plot['total_fight_time_secs'] / 60
df_plot['sig_str_r'] = df_plot['R_avg_SIG_STR_landed']
df_plot['sig_pct_r'] = df_plot['R_avg_SIG_STR_pct']
df_plot['td_land_r'] = df_plot['R_avg_TD_landed']
df_plot['sub_att_r'] = df_plot['R_avg_SUB_ATT']
df_plot = df_plot.sort_values('date').reset_index(drop=True)

# 3. Divisiones
divisiones_hombre = [
    'Heavyweight', 'Light Heavyweight', 'Middleweight',
    'Welterweight', 'Lightweight', 'Featherweight',
    'Bantamweight', 'Flyweight',
]
divisiones_mujer = [
    "Women's Strawweight", "Women's Flyweight",
    "Women's Bantamweight", "Women's Featherweight",
]

colores_div = {
    'Heavyweight'          : '#2C2C2A',
    'Light Heavyweight'    : '#5F5E5A',
    'Middleweight'         : '#D85A30',
    'Welterweight'         : '#378ADD',
    'Lightweight'          : '#1D9E75',
    'Featherweight'        : '#9B59B6',
    'Bantamweight'         : '#F39C12',
    'Flyweight'            : '#E24B4A',
    "Women's Strawweight"  : '#E24B4A',
    "Women's Flyweight"    : '#F39C12',
    "Women's Bantamweight" : '#9B59B6',
    "Women's Featherweight": '#1D9E75',
}

# 4. Función de traces
def build_traces(df, divisiones, col_division, col_sel, ventana, stack_id, show_legend):
    traces = []
    for div in divisiones:
        grupo = df[df[col_division] == div].copy()
        if grupo.empty:
            continue

        grupo = grupo.dropna(subset=[col_sel])
        grupo = grupo.sort_values('date').reset_index(drop=True)
        grupo['n_pelea']    = grupo.index + 1
        grupo['val_smooth'] = grupo[col_sel].rolling(
            window=ventana, min_periods=5
        ).mean()
        grupo = grupo.dropna(subset=['val_smooth'])

        if grupo.empty:
            continue

        color = colores_div.get(div, '#888780')

        traces.append(go.Scatter(
            x             = grupo['n_pelea'],
            y             = grupo['val_smooth'],
            name          = div,
            mode          = 'lines',
            line          = dict(width=0.5, color=color),
            fill          = 'tonexty',
            fillcolor     = color + '55',
            stackgroup    = stack_id,
            legendgroup   = div,
            showlegend    = show_legend,
            hovertemplate = (
                f'<b>{div}</b><br>'
                'Pelea #%{x}<br>'
                f'{label_sel}: ' + '%{y:.2f}'
                '<extra></extra>'
            ),
        ))
    return traces

# 5. Subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Hombres', 'Mujeres'],
    vertical_spacing=0.12,
)

config_generos = [
    {'label': 'Male',   'divisiones': divisiones_hombre, 'row': 1, 'stack': 'stack_1', 'legend': True},
    {'label': 'Female', 'divisiones': divisiones_mujer,  'row': 2, 'stack': 'stack_2', 'legend': False},
]

for cfg in config_generos:
    df_gen = df_plot[df_plot[col_genero] == cfg['label']]
    traces = build_traces(
        df_gen, cfg['divisiones'], col_division,
        col_sel, ventana, cfg['stack'], cfg['legend']
    )
    for t in traces:
        fig.add_trace(t, row=cfg['row'], col=1)

# 6. Layout
fig.update_layout(
    title=dict(
        text=(
            f'Densidad de <b>{label_sel}</b> por género y división'
            f'<br><sup>Áreas apiladas · Promedio móvil {ventana} peleas · Eje X = peleas cronológicas</sup>'
        ),
        x=0.5, font=dict(size=14),
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.15,
        xanchor='center', x=0.5,
        font=dict(size=10), tracegroupgap=8,
    ),
    hovermode     = 'x unified',
    paper_bgcolor = 'rgba(0,0,0,0)',
    plot_bgcolor  = 'rgba(0,0,0,0)',
    height        = 700,
    margin        = dict(t=110, b=120, l=60, r=100),
)
fig.update_xaxes(
    title_text='Número de pelea (cronológico por división)',
    gridcolor='rgba(128,128,128,0.15)', zeroline=False,
)
fig.update_yaxes(
    title_text=label_sel,
    gridcolor='rgba(128,128,128,0.15)', zeroline=False,
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import plotly.figure_factory as ff
import pandas as pd
import numpy as np

# 1. Selector
tipo_odds = 'R_odds'  # opciones: 'R_odds', 'r_dec_odds', 'r_sub_odds', 'r_ko_odds'

odds_config = {
    'R_odds'    : 'Odds generales',
    'r_dec_odds': 'Odds decisión',
    'r_sub_odds': 'Odds sumisión',
    'r_ko_odds' : 'Odds KO/TKO',
}
label_odds = odds_config[tipo_odds]

# 2. Atributos a correlacionar
atributos = {
    'R_avg_SIG_STR_landed' : 'Sig. Strikes lanzados',
    'R_avg_SIG_STR_pct'    : 'Precisión strikes',
    'R_avg_TD_landed'      : 'Takedowns lanzados',
    'R_avg_TD_pct'         : 'Precisión TD',
    'R_avg_SUB_ATT'        : 'Intentos sumisión',
    'R_Height_cms'         : 'Altura (cm)',
    'R_Reach_cms'          : 'Alcance (cm)',
    'R_age'                : 'Edad',
    'R_current_win_streak' : 'Racha victorias',
    'R_current_lose_streak': 'Racha derrotas',
}

# 3. Construir df de correlación
cols_needed = [tipo_odds] + list(atributos.keys())
df_corr = df_ufc[cols_needed].copy().dropna()

# Calcular correlaciones contra las odds
correlaciones = df_corr.corr()[tipo_odds].drop(tipo_odds)
correlaciones.index = [atributos[c] for c in correlaciones.index]
correlaciones = correlaciones.sort_values()

# 4. Colores por dirección y fuerza
def get_color(val):
    if val >= 0.3:   return '#1D9E75'
    elif val >= 0.1: return '#1D9E75AA'
    elif val >= 0:   return '#1D9E7555'
    elif val >= -0.1:return '#D85A3055'
    elif val >= -0.3:return '#D85A30AA'
    else:            return '#D85A30'

colores = [get_color(v) for v in correlaciones.values]

# 5. Figura principal — barras horizontales de correlación
fig = go.Figure()

fig.add_trace(go.Bar(
    x             = correlaciones.values,
    y             = correlaciones.index,
    orientation   = 'h',
    marker=dict(
        color = colores,
        line  = dict(width=0.5, color='rgba(0,0,0,0.2)'),
    ),
    customdata    = correlaciones.values,
    hovertemplate = (
        '<b>%{y}</b><br>'
        f'Correlación con {label_odds}: ' + '%{x:.3f}'
        '<extra></extra>'
    ),
))

# Línea de cero
fig.add_vline(x=0, line_color='rgba(128,128,128,0.5)', line_width=1)

# Zonas de referencia
for x0, x1, label, color in [
    (-1.0, -0.3, 'Correlación<br>fuerte negativa', 'rgba(216,90,48,0.06)'),
    (-0.3, -0.1, 'Moderada<br>negativa',           'rgba(216,90,48,0.03)'),
    ( 0.1,  0.3, 'Moderada<br>positiva',           'rgba(29,158,117,0.03)'),
    ( 0.3,  1.0, 'Correlación<br>fuerte positiva', 'rgba(29,158,117,0.06)'),
]:
    fig.add_vrect(
        x0=x0, x1=x1,
        fillcolor=color, line_width=0,
        annotation_text=label,
        annotation_position='top',
        annotation_font=dict(size=8, color='gray'),
    )

fig.update_layout(
    title=dict(
        text=(
            f'Correlación entre atributos y {label_odds}'
            '<br><sup>Positivo = odds más altas (underdog) · Negativo = odds más bajas (favorito)</sup>'
        ),
        x=0.5, font=dict(size=14),
    ),
    xaxis=dict(
        title     = 'Coeficiente de correlación de Pearson',
        gridcolor = 'rgba(128,128,128,0.15)',
        zeroline  = False,
        range     = [-1, 1],
        tickvals  = [-1, -0.5, -0.3, -0.1, 0, 0.1, 0.3, 0.5, 1],
    ),
    yaxis=dict(
        gridcolor = 'rgba(128,128,128,0.15)',
        zeroline  = False,
    ),
    paper_bgcolor = 'rgba(0,0,0,0)',
    plot_bgcolor  = 'rgba(0,0,0,0)',
    height        = 500,
    margin        = dict(t=110, b=60, l=160, r=60),
)

fig.show()

In [ ]:
# BONUS — Heatmap completo atributos × atributos + odds
cols_heatmap = [tipo_odds] + list(atributos.keys())
df_heat      = df_ufc[cols_heatmap].dropna()

matriz = df_heat.corr().round(2)
labels = [label_odds] + list(atributos.values())
matriz.index   = labels
matriz.columns = labels

fig2 = go.Figure(go.Heatmap(
    z             = matriz.values,
    x             = labels,
    y             = labels,
    colorscale    = [
        [0.0, '#D85A30'], [0.5, '#F1EFE8'], [1.0, '#1D9E75']
    ],
    zmid          = 0,
    zmin          = -1,
    zmax          = 1,
    text          = matriz.values.round(2),
    texttemplate  = '%{text}',
    textfont      = dict(size=9),
    hovertemplate = '<b>%{y}</b> × <b>%{x}</b><br>Correlación: %{z:.3f}<extra></extra>',
    colorbar      = dict(
        title      = 'Pearson',
        tickvals   = [-1, -0.5, 0, 0.5, 1],
        ticktext   = ['-1', '-0.5', '0', '0.5', '1'],
        thickness  = 12,
        len        = 0.8,
    ),
))

fig2.update_layout(
    title=dict(
        text=f'Heatmap de correlaciones — {label_odds} y atributos',
        x=0.5, font=dict(size=14),
    ),
    paper_bgcolor = 'rgba(0,0,0,0)',
    plot_bgcolor  = 'rgba(0,0,0,0)',
    height        = 620,
    margin        = dict(t=80, b=120, l=160, r=60),
    xaxis         = dict(tickangle=-35, tickfont=dict(size=10)),
    yaxis         = dict(tickfont=dict(size=10)),
)

fig2.show()